# LoopStream on Google Colab

Turn a pre-recorded video into a **24/7 YouTube livestream** using a free cloud runtime.

This is the cloud version of the local LoopStream setup. No PC required — Google Colab runs the same ffmpeg loop engine:

```bash
ffmpeg -re -stream_loop -1 -i <source> … -f flv rtmp://a.rtmp.youtube.com/live2/<stream key>
```


In [ ]:
!ffmpeg -version


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Paste your YouTube Live stream key here
STREAM_KEY = "PASTE_YOUTUBE_STREAM_KEY_HERE"


In [ ]:
# Example for a file in Google Drive:
SOURCE = "/content/drive/MyDrive/your-video.mp4"
# …or for a file uploaded straight into Colab:
# SOURCE = "/content/your-video.mp4"


In [ ]:
import subprocess, time

def start_loop(source: str, key: str, log_path: str = "/content/ffmpeg_loop.log"):
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "info",
        "-re",
        "-stream_loop", "-1",
        "-i", source,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-tune", "zerolatency",
        "-b:v", "4500k",
        "-maxrate", "6000k",
        "-bufsize", "12000k",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "128k",
        "-f", "flv",
        f"rtmp://a.rtmp.youtube.com/live2/{key}",
    ]
    log = open(log_path, "ab")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    return proc, log

proc, logf = start_loop(SOURCE, STREAM_KEY)
print("ffmpeg loop started →", SOURCE)


In [ ]:
while True:
    if proc.poll() is not None:
        print(f"ffmpeg exited with code {proc.returncode}")
        break
    time.sleep(15)


In [ ]:
proc.terminate()
proc.wait()
print("ffmpeg loop stopped")


## Notes

- Colab free tier: max runtime ~12h per session; the kernel may disconnect after that or go idle.
- The runtime must stay connected for the stream to continue.
- For a permanent source, use a file in Google Drive (not in a shared/permission-restricted folder) or upload to `/content/`.
- To restart: stop the loop, update `SOURCE` / `STREAM_KEY`, and run the start cell again.
